# 29. The plasticity probe, Tiny ImageNet only

Does the generator get worse at learning a new task after it has already learned eight?
This notebook answers that by the adapter study's own method: copy the model, teach the
copy something it has never seen, measure, then throw the copy away.

It works through every seed in `SEEDS` in one go. Finished probe runs are saved to R2 as
they complete, so stopping the notebook and starting it again continues from where it
stopped rather than from the beginning. A three-seed pass is roughly nine hours, which is
longer than one Colab session, so expect to run the grid cell more than once.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")
if IN_COLAB:
    REPO = Path("/content/path-probe")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")
for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.probe import (RandomLabels, best_of_grid, noise_floor, probe_once,
                         relative_contribution, snapshot, unchanged)

sys.path.insert(0, str(REPO / 'scripts'))
try:
    from dashboard import Dashboard
except ImportError as problem:
    print('Progress panel unavailable, using plain bars:', problem)

    class Dashboard:
        """Just enough of the panel to keep the notebook running."""

        def __init__(self, *args, **kwargs): pass
        def __enter__(self): return self
        def __exit__(self, *args): return False
        def note(self, text): print(text, flush=True)

        def bar(self, total, description, unit='step'):
            from uncle.telemetry import progress_bar
            return progress_bar(total, description)

import torch
HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)

## 2. Settings

The defaults follow the adapter study where this framework allows it. The cell prints how
many probe runs that implies and roughly how long they take, so the budget is visible
before anything starts.

To shorten a first pass, drop `PROBE_TASKS` to one task or `RATES` to one rate. Do not
change `STEPS` between probe points: that would measure the budget instead of the model.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"
ORDER = "order_01"
# Every seed the notebook should cover. It works through them in order and
# saves after each probe run, so a disconnected session resumes where it
# stopped instead of starting again.
SEEDS = [0, 1, 2]

# Where the part-finished results live. They go to R2 rather than the Colab
# disk, because that disk disappears when the session ends. Delete this file
# to force a clean rerun.
RESUME_KEY = f"uncle/plasticity_probe/{STUDY_ID}/{ORDER}/partial.jsonl"
SAVE_EVERY = 10               # upload after this many new runs

# Tasks the stream never saw. Order 01 learns 3, 0, 9, 5, 17, 1, 7 and 14, so
# twelve of the twenty remain. These three are the controls the conclusion
# report already used.
PROBE_TASKS = ["2", "4", "6"]

# Images for the random-label probe, taken from another unseen task so its
# images are not reused by E2.
RANDOM_LABEL_TASK = "8"
RANDOM_LABEL_SEED = 1020      # fixes the wrong labels, shared by every point

# Memorising random labels needs many passes over few images. The first run of
# this notebook used the task's full 5,000 images for 400 steps, which is about
# 5 passes, and the loss never fell below chance: the probe measured nothing.
# These are the adapter study's own dose settings, chosen after it hit the same
# problem: 2,000 images for 600 steps, about 19 passes.
RANDOM_LABEL_IMAGES = 2000

# A model late in a stream may need a different rate than a fresh one, so the
# probe tries a grid and keeps the best. A single fixed rate would confuse a
# rate mismatch with plasticity loss.
RATES = [1e-4, 3e-4, 1e-3]

STEPS = 250                   # natural probe tasks, as the study uses
RANDOM_LABEL_STEPS = 600      # about 19 passes over 2,000 images
BATCH_SIZE = 64
FEATURE_IMAGES = 2000         # for the effective rank

# More repetitions at t = 0, because its spread is what the noise floor is
# built from. The study uses 5 and 2.
REPETITIONS_AT_ZERO = 5
REPETITIONS_LATER = 2

# Which parts of the model each arm lets train: (name, generator frozen, code trains).
# Drop the third entry to go back to the adapter study's two arms and save a third
# of the running time.
ARMS = [
    ("full",           False, True),    # generator and code both train
    ("code-only",      True,  True),    # generator frozen, only the 32 numbers train
    ("generator-only", False, False),   # code held at its random start
]

SAVE_TO_R2 = True
RELEASE_GPU_WHEN_DONE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"

runs_at = lambda reps: (len(PROBE_TASKS) + 1) * len(RATES) * reps * len(ARMS)
TOTAL_RUNS = (runs_at(REPETITIONS_AT_ZERO) + runs_at(REPETITIONS_LATER)) * len(SEEDS)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print(f"Probe runs: {TOTAL_RUNS}  "
      f"({len(PROBE_TASKS)} natural tasks + 1 random-label, {len(RATES)} rates, "
      f"{len(ARMS)} arms, {REPETITIONS_AT_ZERO} and {REPETITIONS_LATER} repetitions, "
      f"{len(SEEDS)} seeds)")
print(f"At roughly 0.6 minutes each that is about {TOTAL_RUNS * 0.6 / 60:.1f} hours.")

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Load the images and the two models

For each seed, `t = 8` is that seed's finished source model and `t = 0` is a generator
built with the same settings and the same seed, which has learned nothing. The models are
built one seed at a time, so only one is ever in memory.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
plan = json.loads(store.download(PREFIX + "/study.json", CACHE / "cache").read_text())
storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = list(plan["orders"][ORDER])
CLASSES = plan["config"]["classes_per_task"]

overlap = set(PROBE_TASKS + [RANDOM_LABEL_TASK]) & set(ORDER_TASKS)
if overlap:
    raise ValueError(f"These probe tasks are in the learning order: {sorted(overlap)}")

needed = tuple(ORDER_TASKS + PROBE_TASKS + [RANDOM_LABEL_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=needed, download=True)
print("Loaded images for", len(needed), "tasks")

def source_state(seed):
    """That seed's finished eight-task model, read from R2."""
    job = next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{seed}")
    state = storage.load(job)
    if state is None or state["report"]["status"] != "complete":
        raise RuntimeError(f"Source model for seed {seed} is not finished in R2.")
    return state

# Check every seed before any training starts, so a missing source model stops
# the notebook now rather than two hours into the run.
for seed in SEEDS:
    source_state(seed)
print("Source models present for seeds:", SEEDS)

def build(seed, restore_source):
    """A model at one probe point. restore_source False gives the t = 0 model."""
    config = Config(**{**plan["config"], "seed": seed, "device": DEVICE,
                       "requests": tuple(("learn", t) for t in ORDER_TASKS)})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    if restore_source:
        checkpoint.restore(source_state(seed), hypernet=hypernet, uncle=trainer)
    return target, hypernet, trainer

# The random labels are drawn once and shared by every probe point. A fresh
# draw per point would make the points incomparable.
from torch.utils.data import Subset
# Fewer images, so the fixed step budget gives enough passes to memorise them.
# The subset is drawn once with a fixed seed and shared by every probe point.
pool = tasks[RANDOM_LABEL_TASK]["train"]
chosen = torch.randperm(len(pool),
                        generator=torch.Generator().manual_seed(RANDOM_LABEL_SEED)
                        )[:RANDOM_LABEL_IMAGES].tolist()
random_label_set = RandomLabels(Subset(pool, chosen), CLASSES, RANDOM_LABEL_SEED)
print(f"Random-label probe: {len(random_label_set)} images, {RANDOM_LABEL_STEPS} steps, "
      f"about {RANDOM_LABEL_STEPS * BATCH_SIZE / len(random_label_set):.0f} passes")
print("Probe tasks:", PROBE_TASKS, "| random labels from task", RANDOM_LABEL_TASK)

## 5. Run the grid

Every probe run snapshots the model, trains a fresh code on a probe task, measures, then
puts the model back. The cell checks after each run that the model really is unchanged, and
stops if it is not.

Each run uses one of three arms. The arm decides which parts are allowed to train:

| Arm | Generator | 32-number task code | What it tells you |
| --- | --- | --- | --- |
| full | trains | trains | how well the model learns a new task |
| code-only | frozen | trains | whether the code alone can learn a task |
| generator-only | trains | frozen at its random start | whether training the code buys anything |

The first two are the adapter study's own arms. The third is the mirror of the second and
answers a separate open question, at the cost of half again as much time.

**This cell can be stopped and started again.** Finished runs are written to a file in R2
every few runs. Running the cell again reads that file back, skips what is already done,
and continues. A session that disconnects after two hours therefore costs nothing.

In [ ]:
import time

# ---- resume -----------------------------------------------------------------
# A run is identified by everything that makes it different from another run.
# Two runs sharing an identity would be measuring the same thing twice.
def identity(run):
    return (run["seed_label"], run["point"], run["endpoint"], run["task"],
            run["learning_rate"], run["arm"], run["repetition"])

PARTIAL = CACHE / "diagnostics" / "plasticity_probe_partial.jsonl"
PARTIAL.parent.mkdir(parents=True, exist_ok=True)

results = []
if store.info(RESUME_KEY) is not None:
    # One JSON object per line, so a half-written final line can be dropped
    # without losing everything before it.
    saved = store.download(RESUME_KEY, CACHE / "cache")
    for line in saved.read_text(encoding="utf-8").splitlines():
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            print("Skipped one unreadable line at the end of the saved file.")
    print(f"Resuming: {len(results)} runs already finished.")
else:
    print("Starting fresh: nothing saved in R2 yet.")

PARTIAL.write_text("".join(json.dumps(r) + "\n" for r in results), encoding="utf-8")
done = {identity(r) for r in results}

def save_partial():
    store.upload(PARTIAL, RESUME_KEY, overwrite=True)

# ---- the grid ---------------------------------------------------------------
new_runs = 0

def grid_at(seed, point, restore_source, repetitions, board):
    """Every probe task, rate, arm and repetition at one probe point."""
    global new_runs
    jobs_here = [(task, tasks[task]["train"], tasks[task]["test"], STEPS, "E2")
                 for task in PROBE_TASKS]
    jobs_here.append((RANDOM_LABEL_TASK, random_label_set, None,
                      RANDOM_LABEL_STEPS, "E1"))

    # Work out what is left before building anything, so a probe point that is
    # already finished costs no model loading at all.
    todo = [(task, train_set, test_set, steps, endpoint, rate, arm, code_only,
             train_code, repetition)
            for task, train_set, test_set, steps, endpoint in jobs_here
            for rate in RATES
            for arm, code_only, train_code in ARMS
            for repetition in range(repetitions)
            if (seed, point, endpoint, task, rate, arm, repetition) not in done]
    if not todo:
        board.note(f"seed {seed}, t = {point}: already finished, skipping")
        return

    target, hypernet, trainer = build(seed, restore_source)
    reference = snapshot(hypernet, trainer)
    bar = board.bar(len(todo), f"seed {seed} t={point}", unit="run")
    for (task, train_set, test_set, steps, endpoint, rate, arm, code_only,
         train_code, repetition) in todo:
        board.note(f"seed {seed} {endpoint} task {task} lr {rate:g} "
                   f"{arm} rep {repetition + 1}")
        started = time.perf_counter()
        run = probe_once(
            hypernet, target, trainer, task=task,
            train_set=train_set, test_set=test_set, steps=steps,
            learning_rate=rate, batch_size=BATCH_SIZE,
            code_only=code_only, train_code=train_code,
            # This seed fixes the probe itself. It depends on the probe task
            # and the repetition, never on the probe point or the run's seed,
            # so two points differ only by the model's state.
            seed=12345 + 1000 * repetition + int(task),
            feature_images=FEATURE_IMAGES if endpoint == "E2" else 0)
        if not unchanged(hypernet, reference):
            raise RuntimeError("A probe changed the model. Stop and fix "
                               "this before trusting anything.")
        run.update(point=point, endpoint=endpoint, repetition=repetition,
                   seed_label=seed, minutes=(time.perf_counter() - started) / 60)
        results.append(run)
        done.add(identity(run))
        # Append rather than rewrite, so saving stays cheap as the file grows.
        with PARTIAL.open("a", encoding="utf-8") as stream:
            stream.write(json.dumps(run) + "\n")
        new_runs += 1
        if new_runs % SAVE_EVERY == 0:
            save_partial()
        bar.update(1)
    bar.close()

    del trainer, hypernet, target
    import gc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    save_partial()

began = time.perf_counter()
with Dashboard(f"Plasticity probe, order {ORDER}") as board:
    for seed in SEEDS:
        grid_at(seed, 0, restore_source=False,
                repetitions=REPETITIONS_AT_ZERO, board=board)
        grid_at(seed, 8, restore_source=True,
                repetitions=REPETITIONS_LATER, board=board)
save_partial()

SESSION_MIN = (time.perf_counter() - began) / 60
# Total time across every session, from the per-run timings, so a resumed run
# does not report only the minutes of its last session.
ELAPSED_MIN = sum(r.get("minutes", 0.0) for r in results)
print(f"\n{len(results)} probe runs in total, {new_runs} of them in this session "
      f"({SESSION_MIN:.1f} minutes).")
print("The model was verified unchanged after every one.")
missing = [(s, p) for s in SEEDS for p in (0, 8)
           if not any(r["seed_label"] == s and r["point"] == p for r in results)]
if missing:
    print("Still to do, as (seed, probe point):", missing)
    print("Run this cell again to continue from here.")

## 6. Pick the best rate, then average

For each seed, probe point, task, arm and repetition, the grid keeps one run: the highest
test accuracy for E2, the lowest final training loss for E1. Then the repetitions are
averaged, and for E2 the three tasks are averaged.

In [ ]:
import statistics

# The arm name the probe records, keyed by the readable name used here.
ARM_KEY = {"full": "full", "code-only": "code_only", "generator-only": "generator_only"}

def selected(seed, point, endpoint, task, arm, repetition):
    """The best rate for one seed, probe point, task, arm and repetition."""
    runs = [r for r in results if r["seed_label"] == seed and r["point"] == point
            and r["endpoint"] == endpoint and r["task"] == task
            and r["arm"] == ARM_KEY[arm] and r["repetition"] == repetition]
    return best_of_grid(runs, endpoint=endpoint) if runs else None

def value_of(run, endpoint):
    return run["test_accuracy"] if endpoint == "E2" else run["final_train_loss"]

def endpoint_values(seed, point, endpoint, arm):
    """One number per repetition: averaged over probe tasks for E2."""
    which = PROBE_TASKS if endpoint == "E2" else [RANDOM_LABEL_TASK]
    reps = REPETITIONS_AT_ZERO if point == 0 else REPETITIONS_LATER
    out = []
    for repetition in range(reps):
        picked = [selected(seed, point, endpoint, task, arm, repetition) for task in which]
        if any(p is None for p in picked):
            continue
        out.append(sum(value_of(p, endpoint) for p in picked) / len(picked))
    return out

mean = lambda values: sum(values) / len(values)

summary = {}
for seed in SEEDS:
    for endpoint in ("E2", "E1"):
        for arm, _, _ in ARMS:
            for point in (0, 8):
                summary[(seed, endpoint, arm, point)] = endpoint_values(
                    seed, point, endpoint, arm)

# A seed counts as finished only when both of its probe points produced values.
SEEDS_DONE = [s for s in SEEDS
              if summary[(s, "E2", "full", 0)] and summary[(s, "E2", "full", 8)]]

WIDTH = 92
def rule(title):
    print(f"\n{title}\n" + "-" * WIDTH)

print("=" * WIDTH)
print(f"PLASTICITY PROBE, order {ORDER}, seeds {SEEDS_DONE}".center(WIDTH))
print(f"{len(results)} runs | {ELAPSED_MIN:.1f} min | Tiny ImageNet probe tasks".center(WIDTH))
print("=" * WIDTH)
if SEEDS_DONE != SEEDS:
    print()
    print("Incomplete. These seeds have no finished pair of probe points yet:",
          [s for s in SEEDS if s not in SEEDS_DONE])
    print("Run the grid cell again to continue. The tables cover the finished seeds.")

rule("TABLE 1. Endpoint values after choosing the best rate, per repetition")
print(f"{'seed':>5}  {'endpoint':<10}{'arm':<16}{'point':>7}{'reps':>6}{'mean':>10}"
      f"{'sd':>9}  values")
print("-" * WIDTH)
for (seed, endpoint, arm, point), values in summary.items():
    if not values:
        continue
    sd = statistics.stdev(values) if len(values) > 1 else float("nan")
    shown = " ".join(f"{v:.2f}" for v in values)
    print(f"{seed:>5}  {endpoint:<10}{arm:<16}{point:>7}{len(values):>6}"
          f"{mean(values):>10.2f}{sd:>9.2f}  {shown}")
print("-" * WIDTH)
print("E2 is test accuracy in percent, averaged over the three probe tasks. Higher is")
print("better. E1 is final training loss on random labels. Lower is better. Chance loss")
print(f"for {CLASSES} classes is ln({CLASSES}) = {__import__('math').log(CLASSES):.3f}.")

## 7. The decision rule

The adapter study declares plasticity loss only when **both** hold:

1. the change from t = 0 has the right sign and its interval excludes zero, and
2. the change is larger than the noise floor.

The noise floor comes from the spread of repetitions at t = 0:

    d = 1.96 * s * sqrt(1/n_t + 1/n_0)

This notebook can check both, because it repeats the probe five times at t = 0. The rule
is applied to each seed on its own, and the seeds are averaged in Table 6. One seed
passing the rule is a reading. The seeds agreeing is a result.

In [ ]:
import math

CHANCE_LOSS = math.log(CLASSES)

# The random-label probe measures nothing until it has actually memorised some
# labels. A final loss sitting at chance means the budget was too small, not
# that capacity is unchanged, so the decision rule must not be applied to it.
# The study requires the probe to fit "well below chance loss".
def e1_worked(seed):
    values = [summary[(seed, "E1", "full", point)] for point in (0, 8)]
    return all(v and mean(v) < CHANCE_LOSS - 0.20 for v in values)

verdicts = {}

for seed in SEEDS_DONE:
    if not e1_worked(seed):
        print()
        print(f"WARNING, seed {seed}. The random-label probe did not memorise. Chance "
              f"loss is {CHANCE_LOSS:.3f} and the full arm ends near it.")
        print("E1 is reported below but means nothing: it measures the budget, not the")
        print("model. Raise RANDOM_LABEL_STEPS or lower RANDOM_LABEL_IMAGES and rerun.")

    rule(f"TABLE 2. The decision rule, full arm, seed {seed}")
    print(f"{'endpoint':<10}{'t=0':>9}{'t=8':>9}{'change':>10}{'floor':>9}"
          f"{'exceeds':>9}  verdict")
    print("-" * WIDTH)
    for endpoint in ("E2", "E1"):
        zero = summary[(seed, endpoint, "full", 0)]
        later = summary[(seed, endpoint, "full", 8)]
        if len(zero) < 2 or not later:
            print(f"{endpoint:<10}  not enough repetitions to decide")
            continue
        mean_zero, mean_later = mean(zero), mean(later)
        change = mean_later - mean_zero
        floor = noise_floor(statistics.stdev(zero), statistics.stdev(zero),
                            len(zero), len(later))
        exceeds = abs(change) > floor
        # E2 is accuracy, so a fall is loss. E1 is loss, so a rise is loss.
        loses = change < 0 if endpoint == "E2" else change > 0
        if endpoint == "E1" and not e1_worked(seed):
            verdict = "VOID, the probe never memorised"
        else:
            verdict = ("PLASTICITY LOSS" if exceeds and loses else
                       "no loss, change is the other way" if exceeds else
                       "NO FINDING, inside the noise floor")
        verdicts[f"seed{seed}|{endpoint}"] = {
            "seed": seed, "t0": mean_zero, "t8": mean_later, "change": change,
            "probe_valid": e1_worked(seed) if endpoint == "E1" else True,
            "noise_floor": floor, "exceeds_floor": exceeds,
            "direction_is_loss": loses, "verdict": verdict}
        print(f"{endpoint:<10}{mean_zero:>9.2f}{mean_later:>9.2f}{change:>+10.2f}"
              f"{floor:>9.2f}{('yes' if exceeds else 'no'):>9}  {verdict}")
    print("-" * WIDTH)
print("The floor is built from the spread of the five repetitions at t = 0, which is")
print("this measurement's own wobble rather than a number borrowed from elsewhere.")

rule("TABLE 3. The code-only control, and what it says about R(t)")
for seed in SEEDS_DONE:
    for endpoint in ("E2", "E1"):
        for point in (0, 8):
            full = summary[(seed, endpoint, "full", point)]
            control = summary[(seed, endpoint, "code-only", point)]
            if not full or not control:
                continue
            gap = mean(full) - mean(control)
            if endpoint == "E1":
                gap = -gap      # for a loss, the control minus the full is the gain
            print(f"  seed {seed} {endpoint} at t={point}: full {mean(full):.2f}, "
                  f"code-only {mean(control):.2f}, generator contributes {gap:+.2f}")
print()
for seed in SEEDS_DONE:
    for endpoint in ("E2", "E1"):
        full0 = summary[(seed, endpoint, "full", 0)]
        ctrl0 = summary[(seed, endpoint, "code-only", 0)]
        full8 = summary[(seed, endpoint, "full", 8)]
        ctrl8 = summary[(seed, endpoint, "code-only", 8)]
        if not all((full0, ctrl0, full8, ctrl8)):
            continue
        try:
            ratio = relative_contribution(mean(full8), mean(ctrl8), mean(full0),
                                          mean(ctrl0), endpoint=endpoint)
            void = endpoint == "E1" and not e1_worked(seed)
            print(f"  seed {seed}: R(8) for {endpoint} = {ratio:.3f}"
                  + ("   (void, the probe never memorised)" if void else ""))
        except ValueError as problem:
            print(f"  seed {seed}: R(8) for {endpoint} is not computable. {problem}")
print()
print("R(t) divides the contribution at t = 8 by the contribution at t = 0. Read the")
print("t = 0 line first. In the adapter study the control there is a linear probe on")
print("pretrained features and scores well, so the denominator is solid. Here the")
print("control at t = 0 is a frozen random generator with a 32-number code. If its")
print("contribution is near zero, R(t) divides by almost nothing and means little.")
print("Report the contributions themselves rather than the ratio when that happens.")

rule("TABLE 4. Feature effective rank, full arm, E2 tasks")
print(f"{'seed':>5}{'point':>8}{'mean rank':>12}{'sd':>9}{'runs':>8}")
print("-" * WIDTH)
for seed in SEEDS_DONE:
    for point in (0, 8):
        reps = REPETITIONS_AT_ZERO if point == 0 else REPETITIONS_LATER
        # Only the runs the grid chose. Averaging over every rate mixes in runs
        # at a rate that did not suit the model, which is most of the spread.
        picked = [selected(seed, point, "E2", task, "full", repetition)
                  for task in PROBE_TASKS for repetition in range(reps)]
        ranks = [r["effective_rank"] for r in picked if r and "effective_rank" in r]
        if ranks:
            sd = statistics.stdev(ranks) if len(ranks) > 1 else float("nan")
            print(f"{seed:>5}{point:>8}{mean(ranks):>12.1f}{sd:>9.1f}{len(ranks):>8}")
print("-" * WIDTH)
print("Effective rank says how many directions the features really use, out of the")
print("layer's full width. A fall over the stream is a known plasticity warning sign.")
print("Probe tasks differ from each other, so a spread of this size across tasks is")
print("expected. Compare the two points, not one number against the layer width.")

rule("TABLE 5. Does training the 32-number task code buy anything?")
print("The full arm trains the generator and the code. The generator-only arm trains")
print("the generator and leaves the code at its random starting value. If the two")
print("score the same, training the code adds nothing in this setup.")
print()
print(f"{'seed':>5}  {'endpoint':<10}{'point':>7}{'full':>10}{'gen-only':>11}{'gap':>9}")
print("-" * WIDTH)
for seed in SEEDS_DONE:
    for endpoint in ("E2", "E1"):
        for point in (0, 8):
            full = summary[(seed, endpoint, "full", point)]
            frozen = summary[(seed, endpoint, "generator-only", point)]
            if not full or not frozen:
                continue
            print(f"{seed:>5}  {endpoint:<10}{point:>7}{mean(full):>10.2f}"
                  f"{mean(frozen):>11.2f}{mean(full) - mean(frozen):>+9.2f}")
print("-" * WIDTH)
print("For E2 the gap is accuracy, so a positive gap means the code helps. For E1 the")
print("gap is loss, so a negative gap means the code helps. Compare the gap with the")
print("noise floor in Table 2 before reading anything into it.")

rule("TABLE 6. Across seeds. This is the headline, not any single seed above")
print(f"{'endpoint':<10}{'seeds':>7}{'mean change':>14}{'sd':>9}  per-seed changes")
print("-" * WIDTH)
across = {}
for endpoint in ("E2", "E1"):
    changes = [verdicts[f"seed{s}|{endpoint}"]["change"]
               for s in SEEDS_DONE if f"seed{s}|{endpoint}" in verdicts]
    if not changes:
        continue
    sd = statistics.stdev(changes) if len(changes) > 1 else float("nan")
    across[endpoint] = {"seeds": len(changes), "mean_change": mean(changes),
                        "sd": sd, "changes": changes}
    shown = " ".join(f"{c:+.2f}" for c in changes)
    print(f"{endpoint:<10}{len(changes):>7}{mean(changes):>+14.2f}{sd:>9.2f}  {shown}")
print("-" * WIDTH)
print("A mean change smaller than the spread between seeds rules out a large plasticity")
print("change. It does not show the change is zero. With three seeds the spread is wide,")
print("so report this as provisional and say how many seeds it rests on.")

## 8. Save

In [ ]:
import csv, datetime, platform

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
seed_label = "seeds" + "".join(str(s) for s in SEEDS_DONE)
EXPORT = CACHE / "diagnostics" / f"plasticity_probe_{ORDER}_{seed_label}_{stamp}"
EXPORT.mkdir(parents=True, exist_ok=True)

record = {
    "notebook": "29_plasticity_probe",
    "protocol": ("Fork and discard, as in reference/adapter-plasticity-study. The model is "
                 "snapshotted, a fresh task code is trained on an unseen task, the result "
                 "is measured, and the model is restored and verified unchanged."),
    "limitations": [
        "Probe tasks come from the same dataset as the stream, so plasticity and "
        "transfer between similar classes are not separated.",
        "Of the adapter study's three natural probe tasks only EuroSAT could fit, "
        "because the generator's output layout is fixed at ten classes.",
        "Two probe points only, t = 0 and t = 8. Mid-stream points need the stream "
        "re-run with probing inline.",
        "Three seeds, two probe points each. The spread between seeds is wider "
        "than the mean change, so this rules out a large change rather than "
        "showing the change is zero.",
    ],
    "code_revision": HEAD, "study_id": STUDY_ID, "order": ORDER,
    "seeds": SEEDS, "seeds_complete": SEEDS_DONE,
    "probe_tasks": PROBE_TASKS, "random_label_task": RANDOM_LABEL_TASK,
    "rates": RATES, "steps": STEPS, "random_label_steps": RANDOM_LABEL_STEPS,
    "batch_size": BATCH_SIZE, "repetitions": {"0": REPETITIONS_AT_ZERO, "8": REPETITIONS_LATER},
    "elapsed_min": ELAPSED_MIN,
    "environment": {"device": DEVICE,
                    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
                    "torch": torch.__version__, "python": sys.version.split()[0],
                    "platform": platform.platform()},
    "verdicts": verdicts,
    "arms": [a for a, _, _ in ARMS],
    "summary": {f"seed{s}|{e}|{a}|t{p}": v for (s, e, a, p), v in summary.items()},
    "across_seeds": across,
    "runs": results,
}
(EXPORT / "probe.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")

with (EXPORT / "runs.csv").open("w", newline="", encoding="utf-8") as stream:
    columns = ["seed_label", "point", "endpoint", "task", "learning_rate", "arm",
               "code_only", "repetition", "minutes",
               "steps", "final_train_loss", "final_train_accuracy", "test_accuracy",
               "test_loss", "effective_rank"]
    writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results)

SAVE_VERIFIED = True
for name in ("probe.json", "runs.csv"):
    path = EXPORT / name
    try:
        text = path.read_text(encoding="utf-8")
        if name.endswith(".json"):
            json.loads(text)
        elif len(text.splitlines()) != len(results) + 1:
            raise ValueError("row count does not match the runs")
        print(f"  verified  {name:<14} {path.stat().st_size:>9,} bytes")
    except Exception as problem:
        SAVE_VERIFIED = False
        print(f"  FAILED    {name:<14} {problem}")

SAVED_LOCATION = str(EXPORT)
print("\nSaved locally:", EXPORT)
if SAVE_TO_R2 and SAVE_VERIFIED:
    destination = PREFIX + "/diagnostics/" + EXPORT.name
    for name in ("probe.json", "runs.csv"):
        store.upload(EXPORT / name, f"{destination}/{name}")
    SAVED_LOCATION = destination
    print("Uploaded to R2:", destination)
elif SAVE_TO_R2:
    print("Not uploading: the local files did not verify.")

## 9. Release the GPU

Only once every file has been written and read back. If anything failed to save the
runtime stays connected, so hours of probing are not lost.

In [ ]:
if not SAVE_VERIFIED:
    print("NOT releasing the GPU: some output files are missing or unreadable.")
    print("Rerun section 8 before closing this runtime, or the probe runs are lost.")
elif SEEDS_DONE != SEEDS:
    print("Results so far saved to:", SAVED_LOCATION)
    print("NOT releasing the GPU. These seeds are not finished yet:",
          [s for s in SEEDS if s not in SEEDS_DONE])
    print("Run the grid cell again to continue from where it stopped.")
elif not RELEASE_GPU_WHEN_DONE:
    print("Results saved to:", SAVED_LOCATION)
    print("RELEASE_GPU_WHEN_DONE is off, so the runtime stays connected.")
elif not IN_COLAB:
    print("Results saved to:", SAVED_LOCATION)
    print("Not running in Colab, so there is no runtime to release.")
else:
    print("Results saved to:", SAVED_LOCATION)
    print("Every seed is finished. Share that path.")
    import gc
    for name in ("tasks", "random_label_set"):
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    from google.colab import runtime
    print("Releasing the Colab runtime now.")
    runtime.unassign()

## 10. What this leaves

Run this for seeds 0, 1 and 2, then the Tiny ImageNet part of the adapter study's
measurement set is done except for three things, and two of those are blocked by this
framework rather than by the data.

| Metric | After this notebook |
| --- | --- |
| Final accuracy, learning accuracy, forgetting, backward transfer, curve slope | Done, notebook 28 |
| E1, random-label loss | Done here |
| E2, natural-task accuracy | Done here, with same-dataset probe tasks |
| Noise floor, and the decision rule | Done here |
| Best-of-grid learning rate | Done here |
| Feature effective rank | Done here |
| Code-only control and the contribution gap | Done here. R(t) itself reported only if its t = 0 denominator is not near zero |
| Whether training the task code buys anything | Done here, Table 5. Not part of the adapter study |
| Joint upper bound | Blocked. It needs one model over all 80 classes, and each task gets its own ten-output classifier |
| Adapter geometry | Not applicable. There is no adapter |
| DTD and Flowers102 probe tasks | Blocked. 47 and 102 classes cannot be generated |

Two pieces of work remain after that, neither of them a new metric:

- **Mid-stream probe points.** t = 2, 4 and 6 need the stream re-run with probing inline,
  because the study kept no mid-stream checkpoints.
- **Orders 02 and 03.** Without them, task identity and position stay confounded, so
  notebook 28's falling curve cannot be read.